<a href="https://colab.research.google.com/github/tallekar/AI-Powered-Human-Detection-Tracking-and-Entry-Analytics-System/blob/main/Gender_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install librosa soundfile scikit-learn matplotlib seaborn


In [ ]:
import pandas as pd

df = pd.read_csv('/content/voice.csv')

df['label'] = df['label'].map({'male': 1, 'female': 0})
df.head()

,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,1
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,1
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,1
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,1
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,1


In [ ]:
import librosa
import numpy as np

def extract_features(file_path):
    audio, sr = librosa.load(file_path, sr=None)

    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
    mfccs_mean = np.mean(mfccs.T, axis=0)

    return mfccs_mean

In [ ]:
import os
import numpy as np # numpy is implicitly available from librosa, but explicitly importing is good practice

X = []
y = []

# Original `dataset_path = "datase/content/voice.csv"` had a typo ('datase')
# and pointed to a file, not a directory.
# This variable should ideally point to a directory that contains 'male' and 'female' subfolders,
# where the actual audio files are located for feature extraction.
# Since no such directory is provided in the environment,
# we use a placeholder path here. You would replace this with the actual path
# to your audio dataset if you have one.
dataset_path = "/content/audio_dataset" # Placeholder for the audio dataset directory

for gender in ["male", "female"]:
    folder_path = os.path.join(dataset_path, gender)

    # Check if the folder exists, as 'dataset_path' is a placeholder.
    # In a real scenario, these folders should contain your audio files.
    if not os.path.exists(folder_path):
        print(f"Warning: Directory '{folder_path}' does not exist. Skipping feature extraction for {gender}.")
        continue # Skip if the directory does not exist

    # The original line 'for file in os.listdir(/content/voice.csv):' caused a SyntaxError
    # because '/content/voice.csv' was missing quotes.
    # Additionally, 'os.listdir()' expects a directory path, not a file path.
    # We fix this by ensuring the path is a string (by using 'folder_path')
    # and by calling os.listdir on the intended directory for each gender.
    for file_name in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file_name)

        # Optional: Add a check for file extension to ensure only audio files are processed
        if not (file_name.lower().endswith(('.wav', '.mp3', '.flac'))):
            continue

        # 'extract_features' function is defined in cell cT5QCjA8O0mj
        features = extract_features(file_path)
        X.append(features)

        if gender == "male":
            y.append(1)
        else:
            y.append(0)

X = np.array(X)
y = np.array(y)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib
import numpy as np # Import numpy for dummy data

# Check if X and y are empty and create dummy data if they are
if X.size == 0 or y.size == 0:
    print("X or y are empty. Generating dummy data for demonstration.")
    # Create some dummy data: e.g., 10 samples, 20 features (matching expected MFCCs)
    X = np.random.rand(10, 20)
    # Create dummy labels: 5 male (1), 5 female (0)
    y = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])


# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Model
model = LogisticRegression()
model.fit(X_train, y_train)

# Save
joblib.dump(model, "gender_model.pkl")
joblib.dump(scaler, "scaler.pkl")

X or y are empty. Generating dummy data for demonstration.


['scaler.pkl']

In [ ]:
import joblib

model = joblib.load("gender_model.pkl")
scaler = joblib.load("scaler.pkl")

def predict_gender(file_path):
    features = extract_features(file_path)
    features = scaler.transform([features])

    prediction = model.predict(features)[0]

    if prediction == 1:
        return "Male"
    else:
        return "Female"

In [ ]:
!pip install gradio



In [ ]:
import gradio as gr

def gradio_predict(audio):
    return predict_gender(audio)

interface = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Audio(type="filepath"),
    outputs="text",
    title="Gender Recognition from Voice"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6eeb6d18834f8b5573.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
